# ResolveAI — QLoRA training on Google Colab

Fine-tunes `Qwen/Qwen2.5-3B-Instruct` with QLoRA on the ResolveAI complaint-classification dataset. Run top-to-bottom on any Colab GPU runtime.

## Before you start
1. Upload the `fine_tuning/` folder from your local repo to Google Drive at `MyDrive/resolveai/fine_tuning/` (drag-and-drop the whole folder into the Drive web UI).
2. **Runtime → Change runtime type → GPU**. Any tier works; the hardware-adaptive cell below picks the right dtype.
3. **Pro only:** Runtime → Manage sessions → enable **background execution** so training survives a closed tab.

**Expected wall time:** ~3h on T4, ~2h on L4, ~30-45 min on A100.  
**Expected cost (Pro):** 5-15 compute units out of 100 monthly.  
**Output:** ~70 MB LoRA adapter at `MyDrive/resolveai/checkpoints/resolveai-sentiment-lora/`.

In [ ]:
# 1. Confirm GPU runtime + see what we got
!nvidia-smi

## 2. Mount Google Drive

Checkpoints land on Drive so a Colab disconnection mid-training doesn't lose progress. With `save_strategy: steps / save_steps: 100`, you can resume from the most recent checkpoint by re-running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Workspace + dependencies

Verifies your upload, creates the checkpoint/results directories on Drive, installs the QLoRA stack.

In [ ]:
import os

PROJECT = '/content/drive/MyDrive/resolveai'
FT_DIR = f'{PROJECT}/fine_tuning'
CHECKPOINT_DIR = f'{PROJECT}/checkpoints/resolveai-sentiment-lora'
RESULTS_DIR = f'{PROJECT}/results'
CFG_PATH = f'{FT_DIR}/configs/training_config.yaml'

assert os.path.isdir(FT_DIR), (
    f'Missing {FT_DIR}. Upload your local fine_tuning/ folder to Drive first.'
)
for required in ('data/formatted/train.jsonl', 'data/formatted/val.jsonl', 'data/formatted/test.jsonl'):
    p = f'{FT_DIR}/{required}'
    assert os.path.isfile(p), f'Missing {p}. Run 02_format_training_data.py locally and re-upload.'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

%pip install -q -U transformers peft trl bitsandbytes accelerate datasets pyyaml matplotlib

## 4. Hardware-adaptive config rewrite

T4 / V100 have **no bf16 tensor cores** so we must use fp16. L4 / A100 / H100 do bf16 natively — use it (larger exponent range, no GradScaler hassles). We also override `output_dir` so checkpoints write to Drive instead of ephemeral Colab disk.

Skip this cell if you've already customized the YAML manually.

In [ ]:
import yaml, subprocess

with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['training']['output_dir'] = CHECKPOINT_DIR

gpu_name = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']
).decode().strip()
print(f'Detected GPU: {gpu_name}')

ampere_or_newer = any(arch in gpu_name for arch in ('A100', 'A10', 'L4', 'L40', 'H100'))
if ampere_or_newer:
    print('-> switching to bf16 (Ampere/Ada/Hopper)')
    cfg['training']['fp16'] = False
    cfg['training']['bf16'] = True
    cfg['quantization']['bnb_4bit_compute_dtype'] = 'bfloat16'
else:
    print('-> keeping fp16 (T4/V100 — no bf16 tensor cores)')
    cfg['training']['fp16'] = True
    cfg['training']['bf16'] = False
    cfg['quantization']['bnb_4bit_compute_dtype'] = 'float16'

with open(CFG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print()
print('Final config:')
print(f"  output_dir:    {cfg['training']['output_dir']}")
print(f"  fp16 / bf16:   {cfg['training']['fp16']} / {cfg['training']['bf16']}")
print(f"  compute_dtype: {cfg['quantization']['bnb_4bit_compute_dtype']}")
print(f"  class_weights enabled: {cfg.get('class_weights', {}).get('enabled', False)}")

## 5. Smoke test the harness (no GPU work yet)

Validates the data path, reports the class-weight derivation on the real distribution, and confirms the JSONL records have the expected `messages` key. Takes ~5 seconds. If this fails, fix it before booking a 3-hour training run.

In [ ]:
!python {FT_DIR}/03_train_qlora.py --dry-run --config {CFG_PATH}

## 6. Train

**This is the long cell.** ~2.5-3h on T4, less on L4/A100. Progress logs every 10 steps (loss), eval every 100 steps (eval_loss). The trainer keeps the 3 most recent + best-by-eval-loss checkpoints (`save_total_limit: 3`) and reloads the best one at the end (`load_best_model_at_end: true`).

**If Colab disconnects:** re-run from this cell. TRL's `resume_from_checkpoint=True` will resume from the latest Drive-persisted checkpoint.

In [ ]:
!python {FT_DIR}/03_train_qlora.py --config {CFG_PATH}

## 7. Verify the saved adapter

Expect `adapter_config.json`, `adapter_model.safetensors`, the tokenizer files, and `training_args.bin`. Total ~70 MB.

In [ ]:
!ls -lh {CHECKPOINT_DIR}
!echo '---'
!du -sh {CHECKPOINT_DIR}

## 8. Evaluate — fine-tuned adapter

Runs the full eval harness against the test split: per-class P/R/F1, urgency MAE/Spearman, % valid JSON, confusion matrices (PNG), latency, and the audit-driven length-bucketed accuracy. ~5-10 min on T4 over 995 test examples.

Output lands in `MyDrive/resolveai/results/`.

In [ ]:
!python {FT_DIR}/04_evaluate.py \
    --config {CFG_PATH} \
    --adapter-dir {CHECKPOINT_DIR} \
    --test-path {FT_DIR}/data/formatted/test.jsonl \
    --output-dir {RESULTS_DIR}

## 9. Evaluate — baseline (bare base model)

Same harness, `--no-adapter` flag. Produces sibling `*_baseline.{json,md,png}` files so you can quote a clean “fine-tuning lifted macro-F1 from X to Y” delta in your portfolio README.

In [ ]:
!python {FT_DIR}/04_evaluate.py \
    --config {CFG_PATH} \
    --no-adapter \
    --test-path {FT_DIR}/data/formatted/test.jsonl \
    --output-dir {RESULTS_DIR}

## 10. Read the eval reports here

Quick sanity — print the markdown reports so you can eyeball the headline numbers without leaving Colab.

In [ ]:
for name in ('eval_report.md', 'eval_report_baseline.md'):
    path = f'{RESULTS_DIR}/{name}'
    if os.path.isfile(path):
        print(f'=== {name} ===\n')
        print(open(path).read())
        print('\n')
    else:
        print(f'(missing: {path})')

## 11. Bring it all back to your local repo

Everything is on Drive. Three options to land it in your local repo at `Resolve_AI/fine_tuning/`:

### Option A — Google Drive for Desktop (recommended)
Install [Google Drive for Desktop](https://www.google.com/drive/download/) on your Mac. Drive contents appear at `~/Library/CloudStorage/GoogleDrive-<email>/My Drive/` (M3 default) or `~/Google Drive/My Drive/`. Then locally:
```bash
DRIVE="$HOME/Library/CloudStorage/GoogleDrive-<your-email>/My Drive/resolveai"
REPO="$HOME/Desktop/Project/Resolve_AI"
cp -R "$DRIVE/checkpoints/resolveai-sentiment-lora" "$REPO/fine_tuning/"
cp -R "$DRIVE/results"                              "$REPO/fine_tuning/"
```

### Option B — Zip + download (no Drive Desktop needed)
Run the next cell. Browser downloads ~70 MB adapter zip + ~1 MB results zip directly.

### Option C — `gdown` from your local terminal
1. In Drive web UI: right-click `checkpoints/resolveai-sentiment-lora/` → Share → “Anyone with the link”.
2. Locally:
   ```bash
   pip install gdown
   gdown --folder "<the share link>" -O ~/Desktop/Project/Resolve_AI/fine_tuning/
   ```
   Don’t forget to revert the share permission afterward.

In [ ]:
# Option B — zip + browser download
import shutil, os
from google.colab import files

adapter_zip = '/content/resolveai-sentiment-lora.zip'
shutil.make_archive(adapter_zip[:-4], 'zip', CHECKPOINT_DIR)
print(f'adapter zip: {os.path.getsize(adapter_zip) / 1e6:.1f} MB')

results_zip = '/content/results.zip'
shutil.make_archive(results_zip[:-4], 'zip', RESULTS_DIR)
print(f'results zip: {os.path.getsize(results_zip) / 1e6:.2f} MB')

files.download(adapter_zip)
files.download(results_zip)

## Done

Once the adapter + results are in your local repo under `Resolve_AI/fine_tuning/`, next steps are:

1. `05_export_gguf.py` — merge LoRA into base, convert to GGUF Q4_K_M, write a Modelfile for Ollama.
2. `ollama create resolveai-sentiment -f Modelfile` and `ollama run` smoke test.
3. Day 13 backend integration — `classifier.py`, `llm_client.py`, `llmops_tracker.py`, `classification_worker.py`.

Both directories are gitignored, so they won’t accidentally land in commits.